In [47]:
import plotly.graph_objects as go
import numpy as np
import json
import pandas as pd
from scipy.stats import pearsonr

In [48]:
base_path = "results"

models = ["google__embeddinggemma-300m",
         "minishlab__potion-base-8M",
         "intfloat__multilingual-e5-large-instruct",
         "intfloat__multilingual-e5-small",
         "BAAI__bge-m3",
         "Qwen__Qwen3-Embedding-0.6B" ]
dataset = "tatoeba:fin-eng"
path = lambda model : f"{base_path}/{model}/{dataset}/"

scores_path = lambda model: path(model)+"mrr@20__Instruct-Query-template.json"
data_path1 = lambda model: path(model)+"prompt_data_cosine_query_to_prompt.json"
data_path2 = lambda model: path(model)+"prompt_data_cosine_query_to_answer.json"

In [49]:
def pprint(title, pears):
    statistic, p = pears.statistic, pears.pvalue
    BOLD  = "\033[1m"
    RED   = "\033[31m"
    RESET = "\033[0m"

    STAT_LIMIT = 0.5   # bold if |r| exceeds this
    P_THRESHOLD = 0.05 # red if p is below this

    stat_str = f"{BOLD}{statistic:.3f}{RESET}" if abs(statistic) > STAT_LIMIT else f"{statistic:.3f}"
    p_str    = f"{RED}{p:.3f}{RESET}"          if p < P_THRESHOLD               else f"{p:.3f}"

    print(f"{title} \t Pearson r: {stat_str}, p-value: {p_str}")


for model in models:
    print(model)
    with open(scores_path(model), "r") as f:
        scores = json.load(f)
    with open(data_path1(model)) as f:
        data1 = json.load(f)
    with open(data_path2(model)) as f:
        data2 = json.load(f) 
    df_scores = pd.DataFrame.from_dict(scores, orient="index", columns=["score"]) #columns= [f"prompt{i}" for i in range(len(scores.values()))])
    df_scores = df_scores.reset_index().rename(columns={"index": "prompt_text"})
    df_prompt = pd.DataFrame.from_dict(data1).T
    df_answer = pd.DataFrame.from_dict(data2).T
    df_answer = df_answer[["prompt_text", "0.0", "1.0"]]
    df_answer["0.0"] = df_answer["0.0"].apply(lambda x: x["mean"])
    df_answer["1.0"] = df_answer["1.0"].apply(lambda x: x["mean"])
    df_prompt = df_prompt[["prompt_text", "0.0", "1.0"]]
    df_prompt["0.0"] = df_prompt["0.0"].apply(lambda x: x["mean"])
    df_prompt["1.0"] = df_prompt["1.0"].apply(lambda x: x["mean"])
    df_A = pd.merge(df_answer, df_scores, on="prompt_text")
    df_P = pd.merge(df_prompt, df_scores, on="prompt_text")
    #display(df_A)
    pprint("Q(0.0):", pearsonr(pd.to_numeric(df_A["0.0"]).tolist(), pd.to_numeric(df_A["score"]).tolist()))
    pprint("A(1.0):", pearsonr(pd.to_numeric(df_A["1.0"]).tolist(), pd.to_numeric(df_A["score"]).tolist()))
    #pprint("P(0.)", pearsonr(pd.to_numeric(df_P["0.0"]).tolist(), pd.to_numeric(df_A["score"]).tolist()))
    pprint("P(1.0):", pearsonr(pd.to_numeric(df_P["1.0"]).tolist(), pd.to_numeric(df_A["score"]).tolist()))
    print("--------------------------------------------------")


google__embeddinggemma-300m
Q(0.0): 	 Pearson r: 0.694, p-value: 0.000
A(1.0): 	 Pearson r: 0.467, p-value: 0.012
P(1.0): 	 Pearson r: -0.563, p-value: 0.002
--------------------------------------------------
minishlab__potion-base-8M
Q(0.0): 	 Pearson r: 0.867, p-value: 0.000
A(1.0): 	 Pearson r: -0.339, p-value: 0.077
P(1.0): 	 Pearson r: -0.796, p-value: 0.000
--------------------------------------------------
intfloat__multilingual-e5-large-instruct
Q(0.0): 	 Pearson r: 0.904, p-value: 0.000
A(1.0): 	 Pearson r: 0.877, p-value: 0.000
P(1.0): 	 Pearson r: -0.779, p-value: 0.000
--------------------------------------------------
intfloat__multilingual-e5-small
Q(0.0): 	 Pearson r: 0.922, p-value: 0.000
A(1.0): 	 Pearson r: 0.872, p-value: 0.000
P(1.0): 	 Pearson r: -0.717, p-value: 0.000
--------------------------------------------------
BAAI__bge-m3
Q(0.0): 	 Pearson r: 0.765, p-value: 0.000
A(1.0): 	 Pearson r: 0.751, p-value: 0.000
P(1.0): 	 Pearson r: -0.832, p-value: 0.000
-----